### DIGI405 Text Analysis Project Notebook

[0.2.5 - 2025-10-08](https://github.com/polsci/DIGI405-assignments/blob/main/CHANGELOG.md) - optional functionality to run inference on augmented data or data from a CSV  
[0.2.3 - 2025-09-15](https://github.com/polsci/DIGI405-assignments/blob/main/CHANGELOG.md) - metric ordering    
[0.2.2 - 2025-09-08](https://github.com/polsci/DIGI405-assignments/blob/main/CHANGELOG.md) - quality of life improvements, probability labels    
[0.2.1 - 2025-08-19](https://github.com/polsci/DIGI405-assignments/blob/main/CHANGELOG.md) - ensure filtering / grid search handled as expected   
Note: Search notebook for 0.2.1/0.2.2/0.2.3/0.2.5 to find changes if you want to apply to your existing notebook.

## Introduction

You should use this notebook as a starting point for your DIGI405 project. It provides code to select your dataset, and run a complete text classification pipeline with [textplumber](https://geoffford.nz/textplumber/), a package that provides an easy to use interface to methods covered in this course.

**Name:** Xian Zhou  
**Student ID:**  39111583
**Project option:** Sentiment  
**Project submission date:**  05/27/2026

Please also add your name to your notebook filename (where it says 'NAME').

### Notebook structure

Sections 1-4 provide code you should modify or extend. In your report, you can refer to code sections by their section number, eg 2.1.

## 1. Setup

You must select the Python 3.12 kernel to run the code in this notebook. 

In [188]:
from datasets import load_dataset, ClassLabel, DatasetDict
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

from sklearn.feature_selection import SelectKBest, mutual_info_classif, chi2
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import Normalizer
from sklearn.pipeline import FeatureUnion
from sklearn.metrics import confusion_matrix, classification_report

from textplumber.core import *
from textplumber.clean import *
from textplumber.preprocess import *
from textplumber.tokens import *
from textplumber.pos import *
from textplumber.embeddings import *
from textplumber.report import *
from textplumber.store import *
from textplumber.lexicons import *
from textplumber.textstats import *

from imblearn.under_sampling import RandomUnderSampler 

import warnings

# in the interests of readability, ignoring this warning
warnings.filterwarnings("ignore", message="Your stop_words may be inconsistent with your preprocessing")

These settings control the display of Pandas dataframes in the notebook.

In [189]:
pd.set_option('display.max_columns', None) # show all columns
pd.set_option('display.max_colwidth', 500) # increase this to see more text in the dataframe

Get word lists: 
* The stop word list is from NLTK.   
* All of the word lists (including the stop word list) can be used to extract lexicon count features to extract features based on a set of words.

In [190]:
stop_words = get_stop_words()
stop_words_lexicon = {'stop_words': stop_words}
empath_lexicons = get_empath_lexicons()
vader_lexicons = get_sentiment_lexicons()

## 2. Load and inspect data

### 2.1 Choose a dataset and preview the labels

Below you can select a dataset for the assignment. The options are `sentiment`, `essay` and `genre`. Change the value of `dataset_option` below. The datasets available on Huggingface.co will be downloaded automatically and a link provided to the dataset card with more information. The `genre` dataset was distributed with this notebook.   

Note:  The `movie_reviews` dataset is being used to demonstrate the notebook and is not one of your options for the assignment.  

In [191]:
# Choose 'essay', 'sentiment', or 'genre' ('movie_reviews' is just for testing/demonstration)
#dataset_option = 'movie_reviews' 
dataset_option = 'sentiment' 
if dataset_option == 'movie_reviews':
	dataset_name = 'polsci/sentiment-polarity-dataset-v2.0'
	dataset_dir = None
	target_labels = ['neg', 'pos']
	text_column = 'text'
	label_column = 'label'
	train_split_name = 'train'
	test_split_name = 'train'
	print('The movie_reviews is to demonstrate the notebook and is not an assignment option.')
elif dataset_option == 'sentiment':
	dataset_name = 'cardiffnlp/tweet_eval'
	dataset_dir = 'sentiment'
	target_labels = ['negative', 'neutral', 'positive']
	text_column = 'text'
	label_column = 'label'
	train_split_name = 'train'
	test_split_name = 'validation'
	print('You selected the sentiment dataset. Read more about this at https://huggingface.co/datasets/cardiffnlp/tweet_eval')
elif dataset_option == 'essay':
	dataset_name = 'polsci/ghostbuster-essay-cleaned'
	dataset_dir = None
	target_labels = ['claude', 'gpt', 'human']
	text_column = 'text'
	label_column = 'label'
	train_split_name = 'train'
	test_split_name = 'test'
	print('You selected the essay dataset. Read more about this at https://huggingface.co/datasets/polsci/ghostbuster-essay-cleaned')
elif dataset_option == 'genre':
    dataset_name = 'genre'
    dataset_type = 'json'
	# Note: Quality of life improvement for version 0.2.2
    dataset_dir = '/srv/source-data/genre_dataset.json' # if you are running this locally change to the path on your machine
    target_labels = ['Fiction', 'Letter', 'Notice', 'Obituary', 'Poetry or verse', 'Recipe', 'Review']
    text_column = 'text'
    label_column = 'label'
    train_split_name = 'train'
    test_split_name = 'test'
    print('You selected the genre dataset.')
else:
	print('Try again! That was not an option!')

You selected the sentiment dataset. Read more about this at https://huggingface.co/datasets/cardiffnlp/tweet_eval


#### Important notes about specific datasets:

* Make sure you go to the relevant Huggingface page to read more about the [essay](https://huggingface.co/datasets/polsci/ghostbuster-essay-cleaned) and [sentiment](https://huggingface.co/datasets/cardiffnlp/tweet_eval/viewer/sentiment) datasets. Note the sentiment dataset is one subset of the larger 'tweet_eval' dataset.  
* For the *sentiment* dataset, it is challenging to get good accuracy with three classes. If you like you can remove the `neutral` class. There is a cell below that does this for you - don't change the cell above.
* For the *essay* dataset, there are differences in punctuation between classes. You should use `character_replacements = {"’": "'", '“': '"', '”': '"',}` in the `TextCleaner` component in your pipeline to make sure you are not overfitting to a quirk of the data.

This loads the dataset. 

In [192]:
if dataset_option != 'genre': # if loading from huggingface ...
    dataset = load_dataset(dataset_name, data_dir=dataset_dir)
else: # if loading the genre dataset from the provided json file
    dataset = load_dataset(dataset_type, data_files=dataset_dir)
    train_dataset = dataset['train'].filter(lambda example: example['split'] == 'train')
    test_dataset = dataset['train'].filter(lambda example: example['split'] == 'test')
    dataset = DatasetDict({
        'train': train_dataset,
        'test': test_dataset
        })

This cell will show you information on the dataset fields and the splits.

In [193]:
preview_dataset(dataset)

Here is the breakdown of the composition of labels in each data-set split.

In [194]:
# casting label column to ClassLabel if not already
cast_column_to_label(dataset, label_column)
label_names = get_label_names(dataset, label_column)

dfs = {}
for split in dataset.keys():
    dfs[split] = dataset[split].to_pandas()
    dfs[split].insert(1, 'label_name', dfs[split][label_column].apply(lambda x: dataset[split].features[label_column].int2str(x)))
    print('Labels for {}:'.format(split))
    preview_label_counts(dfs[split], label_column, label_names)

Column 'label' is already a ClassLabel.
Labels for train:


,label_name,count
label,,
0,negative,7093
1,neutral,20673
2,positive,17849


Labels for validation:


,label_name,count
label,,
0,negative,312
1,neutral,869
2,positive,819


Labels for test:


,label_name,count
label,,
0,negative,3972
1,neutral,5937
2,positive,2375


### 2.2 Configure the labels (optional)

* You can override the default labels for the data-set here to make the task more or less challenging. High accuracy does not guarantee a high grade. 
* See the assignment instructions and the dataset card or corresponding paper for explanations of the data.  
* Read the comments below and uncomment the relevant lines for your data-set if and amend the label names if needed.
* Remember, this is optional.

In [195]:
# for the movie reviews dataset (this is just for testing/demonstration) - there are 2 labels and that is it!

# for the sentiment dataset - there are 3 labels - you can make the task simpler as a binary classification problem using one of these options:
#target_labels = ['negative', 'neutral']
#target_labels = ['negative', 'positive']
#target_labels = ['neutral', 'positive']

# for the essay dataset - there are 7 labels - you can make the task simpler as a binary classification problem using one of these options:
#target_labels = ['claude', 'gpt']
#target_labels = ['human', 'gpt'] 
#target_labels = ['human', 'claude']

# for the genre dataset - there are 7 labels - you can turn the task into one or more binary classification problems using options such as:
#target_labels = ['Letter', 'Notice']
#target_labels = ['Letter', 'Fiction']
#target_labels = ['Review', 'Fiction']
#target_labels = ['Notice', 'Obituary']

print(target_labels)

['negative', 'neutral', 'positive']


### 2.3 Prepare the train and test splits

* This cell handles the train-test split for you.
* Some of the data-sets are unbalanced. This cell will balance the training data using under-sampling.

In [196]:
target_classes = [label_names.index(name) for name in target_labels]
target_names = [label_names[i] for i in target_classes]

if train_split_name == test_split_name:
    X = dataset[train_split_name].to_pandas()
    X.insert(1, 'label_name', dfs[train_split_name][label_column].apply(lambda x: dataset[train_split_name].features[label_column].int2str(x)))
    y = np.array(dataset[train_split_name][label_column])

    mask = np.isin(y, target_classes)
    X = X.loc[mask]
    y = y[mask]

    # creating df splits with original data first  - so can look at the train data if needed
    dfs['train'], dfs['test'], y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    # we're just using the text for features
    X_train = np.array(dfs['train'][text_column])
    X_test = np.array(dfs['test'][text_column])
else:
    X_train = np.array(dataset[train_split_name][text_column])
    y_train = np.array(dataset[train_split_name][label_column])
    X_test = np.array(dataset[test_split_name][text_column])
    y_test = np.array(dataset[test_split_name][label_column])

    mask = np.isin(y_train, target_classes)
    mask_test = np.isin(y_test, target_classes)

    X_train = X_train[mask]
    y_train = y_train[mask]
    X_test = X_test[mask_test]
    y_test = y_test[mask_test]

# this cell undersamples all but the minority class to balance the training data
X_train = X_train.reshape(-1, 1)
X_train, y_train = RandomUnderSampler(random_state=0).fit_resample(X_train, y_train)
X_train = X_train.reshape(-1)

preview_splits(X_train, y_train, X_test, y_test, target_classes = target_classes, target_names = target_names)

Train: 21279 samples, 3 classes


,label_name,count
0,,
0,negative,7093
1,neutral,7093
2,positive,7093


Test: 2000 samples, 3 classes


,label_name,count
0,,
1,neutral,869
2,positive,819
0,negative,312


### 2.4 Preview the texts

Time to get to know your data. We will only preview the train split.

In [197]:
y_train_names = map(lambda x: label_names[x], y_train)
# Note: Version 0.2.1 corrects display of the dataframe to ensure filtering by the selected labels 
display(dfs['train'][dfs['train']['label_name'].isin(y_train_names)].sample(20))
#orginal code smaple(10), switched to 20 for more info 

,text,label_name,label
11388,What teams do you think may get transferred/moved in the future? I can only see the Sacramento Kings\u002c Phoenix...,neutral,1
19194,@user lmao. I only miss the jokes and what not. Walking in the sun was shit! And that bitch Candy Cane is still missin!,negative,0
40802,Drogba(as reports suggest he is preferred to start)\u002c If Cannavaro has anything close to the game he had on friday then he takes this.,neutral,1
1413,"""Frank Gifford always lived in someone's shadow, whether it was Howard Cosell's on """"Monday Night Football"""" or Kathie Lee's in married life.""",negative,0
37217,@user Hey! if you are looking for new music you may really enjoy my bands material @user (: we sound similar to The Cab!,positive,2
14158,US football down 7-3 at Chardon tonight after 1st Q,neutral,1
12432,@user I\u2019m presently in Poland but will fill you in on my return home to Scotland on Saturday.,neutral,1
11505,@user r yall going to be in dallas friday & saturday?,neutral,1
18161,"""I am going to see Ed Sheeran, live, in the 14th row. I'm in disbelief""",positive,2
140,"""Christianity is nothing else but the imitation of God\u2019s nature. We seek a relationship with God through Jesus ...\""""""""Lord that I may see\"""""""".""",neutral,1


## full text check 
change the selected_index to check with the original training tweet

In [198]:
selected_index = 91
# switch the index number for different comments
preview_row_text(dfs['train'], selected_index, text_column = text_column, limit=400) # change limit to see more of the text if needed

,Value
Attribute,
label_name,neutral
label,1


text:
The Hitchhiker's Guide to the Galaxy Game - 30th Anniversary Edition Need to
sign up with the BBC online.


## 3. Create a classification pipeline and train a model

Create a Sci-kit Learn pipeline to preprocess the texts and train a classification model. The pipeline components will be added in through the notebook. There are a number of pipeline components you can access through the `textplumber` package. You will have an opportunity to learn about this in labs, but documentation is [available here](https://geoffford.nz/textplumber).

To speed up preprocessing some of the pipeline components store the preprocessed data in a cache to avoid recomputing them. Run this as is - it will create an SQLite file with the name of your dataset option in the directory of the notebook. This will speed up some repeated processing (e.g. tokenization with Spacy).

In [199]:
feature_store = TextFeatureStore(f'assignment-{dataset_option}.sqlite')

The pipeline below includes a number of different components. Most are commented out on the first run of the notebook. There are lots of options for each component. You will need to look at the documentation and examples in labs to learn about these. These components can extract different kinds of features, any of which can be applied to build a model. The feature types include:

* Token features  
* Bigram features  
* Parts of speech features
* Lexicon-based features  
* Document-level statistics  
* Text embeddings


## 3.1 caution_stop_words (customised stop words list) 
This is a special word list designed to keep “non-noise” stop words, which helps the model to detect negation. Preserving sentiment-relevant negation forms would be easier for the model to recognise polarity reversal.

In [200]:
caution_stop_words = [
    i for i in stop_words
    if i not in ('not', 'no', 'never', 'nt', "n't","ever","ca")
]

## 3.2.1 Pipeline 1 model design 
Features combination: Token + POS + Lexicon

Algorithmic model: Logistic regression 

In [218]:
pipeline1 = Pipeline([
	('cleaner', TextCleaner(strip_whitespace=True)), # for the essay dataset you should use character_replacements = {"’": "'", '“': '"', '”': '"',}
	('spacy', SpacyPreprocessor(feature_store=feature_store)),
	('features', FeatureUnion([
		('tokens', # token features - these can be single tokens or ngrams of tokens using TokensVectorizer - see textplumber documentation for examples
			Pipeline([
				('spacy_token_vectorizer', 
                 TokensVectorizer(feature_store = feature_store, 
                                  vectorizer_type='tfidf', #change count to tfidf
                                  max_features=700, #change
                                  lowercase = False, #change from true to false 
                                  remove_punctuation = False, #change from true to false
                                  stop_words = caution_stop_words, 
                                  min_df=2, max_df=1.0, #change appears at least 2 tweets, 
                                  ngram_range=(1, 2))),#change
				#('selector', SelectKBest(score_func=mutual_info_classif, k=200)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
        ('pos', # pos features - these can be a single label or ngrams of pos tags using POSVectorizer - see textplumber documentation for examples
		 	Pipeline([
		 		('spacy_pos_vectorizer', POSVectorizer(feature_store=feature_store)),
		 		('selector', SelectKBest(score_func=mutual_info_classif, k=5)),
		 		('scaler', StandardScaler(with_mean=False)),
		 		], verbose = True)),

		
		 ('lexicon', # lexicon features - defined above are empath_lexicons, sentiment_lexicons and stop_words_lexicon - see textplumber documentation for examples
		 	Pipeline([
				('lexicon_vectorizer', LexiconCountVectorizer(feature_store=feature_store, lexicons=vader_lexicons)), # the notebook has already provided example lexicons right at the top!
		  		('selector', SelectKBest(score_func=mutual_info_classif, k=2)), #change, do not keep only 2 lexicon features
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),

		 #('embeddings', Model2VecEmbedder(feature_store=feature_store)), # extract embeddings using Model2Vec - textplumber documentation for examples

		], verbose = True)),
	
	('classifier', LogisticRegression(max_iter=5000, random_state=42)) # for logistic regression - only select one classifier!
    #('classifier', DecisionTreeClassifier(max_depth = 3, random_state=42)) # for decision tree - only select one classifier!
], verbose = True) # using verbose because I like to see what is going on

display(pipeline1)



Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>,...
                                                                                                                'adored',
                                                                                                                'adoring',
                                                                                                                'adorn',
                                                                                                                'adorning',
                                                                                                                'advanced',
                                                                                                                'advantage',
                                                                                                                'advantaged',
                                                                                                                'advantageous',
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f81f530e520>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True))],
                              verbose=True)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))],
         verbose=True)

In [219]:
pipeline1.fit(X_train, y_train)

[Pipeline] ........... (step 1 of 4) Processing cleaner, total=   0.0s
[Pipeline] ............. (step 2 of 4) Processing spacy, total=   1.8s
[Pipeline]  (step 1 of 2) Processing spacy_token_vectorizer, total=   5.0s
[Pipeline] ............ (step 2 of 2) Processing scaler, total=   0.0s
[FeatureUnion] ........ (step 1 of 3) Processing tokens, total=   5.0s
[Pipeline]  (step 1 of 3) Processing spacy_pos_vectorizer, total=   3.8s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ........... (step 2 of 3) Processing pos, total=   3.8s
[Pipeline]  (step 1 of 3) Processing lexicon_vectorizer, total=   4.1s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.1s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ....... (step 3 of 3) Processing lexicon, total=   4.2s
[Pipeline] .......... (step 3 of 4) Processing features, total=  13.1s


Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>,...
                                                                                                                'adored',
                                                                                                                'adoring',
                                                                                                                'adorn',
                                                                                                                'adorning',
                                                                                                                'advanced',
                                                                                                                'advantage',
                                                                                                                'advantaged',
                                                                                                                'advantageous',
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f81f530e520>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True))],
                              verbose=True)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))],
         verbose=True)

In [220]:
y_predicted = pipeline1.predict(X_test)
print(classification_report(y_test, y_predicted, labels = target_classes, target_names = target_names, digits=3, zero_division=0)) # 0.2.3 metric ordering
plot_confusion_matrix(y_test, y_predicted, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.420     0.696     0.524       312
     neutral      0.656     0.543     0.594       869
    positive      0.708     0.659     0.683       819

    accuracy                          0.615      2000
   macro avg      0.594     0.633     0.600      2000
weighted avg      0.640     0.615     0.619      2000



## 3.2.2 Pipeline 2 model design 
Features combination: Token + POS + Lexicon + Embeddings

Algorithmic model: Logistic regression 

In [204]:
pipeline2 = Pipeline([
	('cleaner', TextCleaner(strip_whitespace=True)), # for the essay dataset you should use character_replacements = {"’": "'", '“': '"', '”': '"',}
	('spacy', SpacyPreprocessor(feature_store=feature_store)),
	('features', FeatureUnion([
		('tokens', # token features - these can be single tokens or ngrams of tokens using TokensVectorizer - see textplumber documentation for examples
			Pipeline([
				('spacy_token_vectorizer', 
                 TokensVectorizer(feature_store = feature_store, 
                                  vectorizer_type='tfidf', #change count to tfidf
                                  max_features=700, #change
                                  lowercase = False, #change from true to false 
                                  remove_punctuation = False, #change from true to false
                                  stop_words = caution_stop_words, 
                                  min_df=2, max_df=1.0, #change appears at least 2 tweets, 
                                  ngram_range=(1, 2))),#change
				#('selector', SelectKBest(score_func=mutual_info_classif, k=200)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
        ('pos', # pos features - these can be a single label or ngrams of pos tags using POSVectorizer - see textplumber documentation for examples
		 	Pipeline([
		 		('spacy_pos_vectorizer', POSVectorizer(feature_store=feature_store)),
		 		('selector', SelectKBest(score_func=mutual_info_classif, k=5)),
		 		('scaler', StandardScaler(with_mean=False)),
		 		], verbose = True)),

		
		 ('lexicon', # lexicon features - defined above are empath_lexicons, sentiment_lexicons and stop_words_lexicon - see textplumber documentation for examples
		 	Pipeline([
				('lexicon_vectorizer', LexiconCountVectorizer(feature_store=feature_store, lexicons=vader_lexicons)), # the notebook has already provided example lexicons right at the top!
		  		('selector', SelectKBest(score_func=mutual_info_classif, k=2)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
	 ('embeddings', Model2VecEmbedder(feature_store=feature_store)), # extract embeddings using Model2Vec - textplumber documentation for examples

		], verbose = True)),
	
	('classifier', 
     LogisticRegression(
         max_iter=5000,
         random_state=42)) # for logistic regression - only select one classifier!
    #('classifier', DecisionTreeClassifier(max_depth = 3, random_state=42)) # for decision tree - only select one classifier!
], verbose = True) # using verbose because I like to see what is going on

display(pipeline2)



Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>,...
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f81f530e520>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True)),
                                                ('embeddings',
                                                 Model2VecEmbedder(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>))],
                              verbose=True)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))],
         verbose=True)

In [205]:
pipeline2.fit(X_train, y_train)

[Pipeline] ........... (step 1 of 4) Processing cleaner, total=   0.0s
[Pipeline] ............. (step 2 of 4) Processing spacy, total=   1.8s
[Pipeline]  (step 1 of 2) Processing spacy_token_vectorizer, total=   4.7s
[Pipeline] ............ (step 2 of 2) Processing scaler, total=   0.0s
[FeatureUnion] ........ (step 1 of 4) Processing tokens, total=   4.7s
[Pipeline]  (step 1 of 3) Processing spacy_pos_vectorizer, total=   4.1s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ........... (step 2 of 4) Processing pos, total=   4.1s
[Pipeline]  (step 1 of 3) Processing lexicon_vectorizer, total=   4.1s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.1s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ....... (step 3 of 4) Processing lexicon, total=   4.3s
[FeatureUnion] .... (step 4 of 4) Processing embeddings, total=   2.4s


Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>,...
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f81f530e520>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True)),
                                                ('embeddings',
                                                 Model2VecEmbedder(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>))],
                              verbose=True)),
                ('classifier',
                 LogisticRegression(max_iter=5000, random_state=42))],
         verbose=True)

In [206]:
y_predicted = pipeline2.predict(X_test)
print(classification_report(y_test, y_predicted, labels = target_classes, target_names = target_names, digits=3, zero_division=0)) # 0.2.3 metric ordering
plot_confusion_matrix(y_test, y_predicted, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.442     0.731     0.551       312
     neutral      0.676     0.560     0.613       869
    positive      0.721     0.673     0.696       819

    accuracy                          0.633      2000
   macro avg      0.613     0.655     0.620      2000
weighted avg      0.658     0.633     0.637      2000



## 3.2.3 Pipeline 3 model design 
Features combination: Token + POS + Lexicon

Algorithmic model: Decision tree

In [207]:
pipeline3 = Pipeline([
	('cleaner', TextCleaner(strip_whitespace=True)), # for the essay dataset you should use character_replacements = {"’": "'", '“': '"', '”': '"',}
	('spacy', SpacyPreprocessor(feature_store=feature_store)),
	('features', FeatureUnion([
		('tokens', # token features - these can be single tokens or ngrams of tokens using TokensVectorizer - see textplumber documentation for examples
			Pipeline([
				('spacy_token_vectorizer', 
                 TokensVectorizer(feature_store = feature_store, 
                                  vectorizer_type='tfidf', #change count to tfidf
                                  max_features=700, #change
                                  lowercase = False, #change from true to false 
                                  remove_punctuation = False, #change from true to false
                                  stop_words = caution_stop_words, 
                                  min_df=2, max_df=1.0, #change appears at least 2 tweets, 
                                  ngram_range=(1, 2))),#change
				#('selector', SelectKBest(score_func=mutual_info_classif, k=200)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
        ('pos', # pos features - these can be a single label or ngrams of pos tags using POSVectorizer - see textplumber documentation for examples
		 	Pipeline([
		 		('spacy_pos_vectorizer', POSVectorizer(feature_store=feature_store)),
		 		('selector', SelectKBest(score_func=mutual_info_classif, k=5)),
		 		('scaler', StandardScaler(with_mean=False)),
		 		], verbose = True)),

		
		 ('lexicon', # lexicon features - defined above are empath_lexicons, sentiment_lexicons and stop_words_lexicon - see textplumber documentation for examples
		 	Pipeline([
				('lexicon_vectorizer', LexiconCountVectorizer(feature_store=feature_store, lexicons=vader_lexicons)), # the notebook has already provided example lexicons right at the top!
		  		('selector', SelectKBest(score_func=mutual_info_classif, k=2)), #change, do not keep only 2 lexicon features
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),

		 #('embeddings', Model2VecEmbedder(feature_store=feature_store)), # extract embeddings using Model2Vec - textplumber documentation for examples

		], verbose = True)),
	
	#('classifier', LogisticRegression(max_iter=5000, random_state=42)) # for logistic regression - only select one classifier!
    ('classifier', DecisionTreeClassifier(max_depth = 3, random_state=42)) # for decision tree - only select one classifier!
], verbose = True) # using verbose because I like to see what is going on

display(pipeline3)



Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>,...
                                                                                                                'adored',
                                                                                                                'adoring',
                                                                                                                'adorn',
                                                                                                                'adorning',
                                                                                                                'advanced',
                                                                                                                'advantage',
                                                                                                                'advantaged',
                                                                                                                'advantageous',
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f81f530e520>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True))],
                              verbose=True)),
                ('classifier',
                 DecisionTreeClassifier(max_depth=3, random_state=42))],
         verbose=True)

In [208]:
pipeline3.fit(X_train, y_train)

[Pipeline] ........... (step 1 of 4) Processing cleaner, total=   0.0s
[Pipeline] ............. (step 2 of 4) Processing spacy, total=   1.8s
[Pipeline]  (step 1 of 2) Processing spacy_token_vectorizer, total=   5.1s
[Pipeline] ............ (step 2 of 2) Processing scaler, total=   0.0s
[FeatureUnion] ........ (step 1 of 3) Processing tokens, total=   5.1s
[Pipeline]  (step 1 of 3) Processing spacy_pos_vectorizer, total=   3.8s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ........... (step 2 of 3) Processing pos, total=   3.8s
[Pipeline]  (step 1 of 3) Processing lexicon_vectorizer, total=   4.4s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.1s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ....... (step 3 of 3) Processing lexicon, total=   4.5s
[Pipeline] .......... (step 3 of 4) Processing features, total=  13.4s


Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>,...
                                                                                                                'adored',
                                                                                                                'adoring',
                                                                                                                'adorn',
                                                                                                                'adorning',
                                                                                                                'advanced',
                                                                                                                'advantage',
                                                                                                                'advantaged',
                                                                                                                'advantageous',
                                                                                                                'advantaging', ...]})),
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f81f530e520>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True))],
                              verbose=True)),
                ('classifier',
                 DecisionTreeClassifier(max_depth=3, random_state=42))],
         verbose=True)

In [209]:
y_predicted = pipeline3.predict(X_test)
print(classification_report(y_test, y_predicted, labels = target_classes, target_names = target_names, digits=3, zero_division=0)) # 0.2.3 metric ordering
plot_confusion_matrix(y_test, y_predicted, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.412     0.474     0.441       312
     neutral      0.634     0.484     0.549       869
    positive      0.599     0.714     0.651       819

    accuracy                          0.577      2000
   macro avg      0.548     0.558     0.547      2000
weighted avg      0.585     0.577     0.574      2000



## 3.2.4 Pipeline 4 model design 
Features combination: Token + POS + Lexicon + Embeddings

Algorithmic model: Decision tree

In [210]:
pipeline4 = Pipeline([
	('cleaner', TextCleaner(strip_whitespace=True)), # for the essay dataset you should use character_replacements = {"’": "'", '“': '"', '”': '"',}
	('spacy', SpacyPreprocessor(feature_store=feature_store)),
	('features', FeatureUnion([
		('tokens', # token features - these can be single tokens or ngrams of tokens using TokensVectorizer - see textplumber documentation for examples
			Pipeline([
				('spacy_token_vectorizer', 
                 TokensVectorizer(feature_store = feature_store, 
                                  vectorizer_type='tfidf', #change count to tfidf
                                  max_features=700, #change
                                  lowercase = False, #change from true to false 
                                  remove_punctuation = False, #change from true to false
                                  stop_words = caution_stop_words, 
                                  min_df=2, max_df=1.0, #change appears at least 2 tweets, 
                                  ngram_range=(1, 2))),#change
				#('selector', SelectKBest(score_func=mutual_info_classif, k=200)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
        ('pos', # pos features - these can be a single label or ngrams of pos tags using POSVectorizer - see textplumber documentation for examples
		 	Pipeline([
		 		('spacy_pos_vectorizer', POSVectorizer(feature_store=feature_store)),
		 		('selector', SelectKBest(score_func=mutual_info_classif, k=5)),
		 		('scaler', StandardScaler(with_mean=False)),
		 		], verbose = True)),

		
		 ('lexicon', # lexicon features - defined above are empath_lexicons, sentiment_lexicons and stop_words_lexicon - see textplumber documentation for examples
		 	Pipeline([
				('lexicon_vectorizer', LexiconCountVectorizer(feature_store=feature_store, lexicons=vader_lexicons)), # the notebook has already provided example lexicons right at the top!
		  		('selector', SelectKBest(score_func=mutual_info_classif, k=2)), 
				('scaler', StandardScaler(with_mean=False)),
				], verbose = True)),
	 ('embeddings', Model2VecEmbedder(feature_store=feature_store)), # extract embeddings using Model2Vec - textplumber documentation for examples

		], verbose = True)),
	
	#('classifier', LogisticRegression(max_iter=5000,random_state=42)) # for logistic regression - only select one classifier!
    ('classifier', DecisionTreeClassifier(max_depth = 3, random_state=42)) # for decision tree - only select one classifier!
], verbose = True) # using verbose because I like to see what is going on

display(pipeline4)



Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>,...
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f81f530e520>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True)),
                                                ('embeddings',
                                                 Model2VecEmbedder(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>))],
                              verbose=True)),
                ('classifier',
                 DecisionTreeClassifier(max_depth=3, random_state=42))],
         verbose=True)

In [211]:
pipeline4.fit(X_train, y_train)

[Pipeline] ........... (step 1 of 4) Processing cleaner, total=   0.0s
[Pipeline] ............. (step 2 of 4) Processing spacy, total=   1.8s
[Pipeline]  (step 1 of 2) Processing spacy_token_vectorizer, total=   4.7s
[Pipeline] ............ (step 2 of 2) Processing scaler, total=   0.0s
[FeatureUnion] ........ (step 1 of 4) Processing tokens, total=   4.7s
[Pipeline]  (step 1 of 3) Processing spacy_pos_vectorizer, total=   4.0s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.0s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ........... (step 2 of 4) Processing pos, total=   4.0s
[Pipeline]  (step 1 of 3) Processing lexicon_vectorizer, total=   4.1s
[Pipeline] .......... (step 2 of 3) Processing selector, total=   0.1s
[Pipeline] ............ (step 3 of 3) Processing scaler, total=   0.0s
[FeatureUnion] ....... (step 3 of 4) Processing lexicon, total=   4.2s
[FeatureUnion] .... (step 4 of 4) Processing embeddings, total=   2.4s


Pipeline(steps=[('cleaner', TextCleaner(strip_whitespace=True)),
                ('spacy',
                 SpacyPreprocessor(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>)),
                ('features',
                 FeatureUnion(transformer_list=[('tokens',
                                                 Pipeline(steps=[('spacy_token_vectorizer',
                                                                  TokensVectorizer(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>,...
                                                                 ('selector',
                                                                  SelectKBest(k=2,
                                                                              score_func=<function mutual_info_classif at 0x7f81f530e520>)),
                                                                 ('scaler',
                                                                  StandardScaler(with_mean=False))],
                                                          verbose=True)),
                                                ('embeddings',
                                                 Model2VecEmbedder(feature_store=<textplumber.store.TextFeatureStore object at 0x7f81b78c4d40>))],
                              verbose=True)),
                ('classifier',
                 DecisionTreeClassifier(max_depth=3, random_state=42))],
         verbose=True)

In [212]:
y_predicted = pipeline4.predict(X_test)
print(classification_report(y_test, y_predicted, labels = target_classes, target_names = target_names, digits=3, zero_division=0)) # 0.2.3 metric ordering
plot_confusion_matrix(y_test, y_predicted, target_classes, target_names)

              precision    recall  f1-score   support

    negative      0.436     0.446     0.441       312
     neutral      0.592     0.557     0.574       869
    positive      0.615     0.648     0.631       819

    accuracy                          0.577      2000
   macro avg      0.548     0.550     0.549      2000
weighted avg      0.577     0.577     0.577      2000



## 4. Evaluate your model and investigate model predictions

You already have some metrics in the cell above. Below is some additional reporting to help you understand your model.

### 4.1 Classifier-specific features

If you are using a Decision Tree classifier in your pipeline, this will plot it ...

In [213]:
if pipeline1.named_steps['classifier'].__class__.__name__ == 'LogisticRegression':
	plot_logistic_regression_features_from_pipeline(pipeline1, target_classes, target_names, top_n=20, classifier_step_name = 'classifier', features_step_name = 'features')

,Feature,Log Odds (Logit),Odds Ratio
706,lexicon__negative,0.462260,1.587658
705,lexicon__positive,-0.270950,0.762655
132,tokens__Brock,-0.252321,0.776996
269,tokens__Lesnar,0.246496,1.279534
574,tokens__not,0.198211,1.219220
564,tokens__n't,0.195588,1.216026
91,tokens__:(,0.170549,1.185956
324,tokens__Rollins,-0.169700,0.843918
334,tokens__Saudi Arabia,0.160020,1.173535
116,tokens__Arabia,-0.155325,0.856137


,Feature,Log Odds (Logit),Odds Ratio
705,lexicon__positive,-0.221023,0.801699
197,tokens__Gifford,0.208012,1.231228
187,tokens__Frank Gifford,-0.199393,0.819228
0,tokens__!,-0.185816,0.830426
393,tokens__Wright,0.159061,1.172410
706,lexicon__negative,-0.144893,0.865115
496,tokens__gon na,-0.142327,0.867338
464,tokens__excited,-0.141520,0.868037
702,pos__PROPN,0.141430,1.151920
324,tokens__Rollins,0.136011,1.145694


,Feature,Log Odds (Logit),Odds Ratio
705,lexicon__positive,0.491972,1.635539
0,tokens__!,0.326209,1.385705
706,lexicon__negative,-0.317367,0.728064
116,tokens__Arabia,0.264167,1.302345
393,tokens__Wright,-0.251810,0.777393
159,tokens__David Wright,0.244116,1.276493
334,tokens__Saudi Arabia,-0.242748,0.784469
182,tokens__Foo,0.220842,1.247127
142,tokens__Caitlyn,-0.184739,0.831321
574,tokens__not,-0.183466,0.832380


### 4.2 Investigate correct and incorrect predictions

To see the predictions of your model run this cell. The output can be quite long depending on the dataset and the number of misclassifications. The Pandas `max_rows` is configured at the top of the cell to restrict the length of output. You can adjust this as required. This is reset back to the Pandas default at the end of the cell.

In [233]:
# adjust max rows
pd.set_option('display.max_rows', 10) # show all rows

# creating dataframe from y_predicted, y_test and the text
predictions_df = pd.DataFrame(data = {'true': y_test, 'predicted': y_predicted})
y_predicted_probs = pipeline1.predict_proba(X_test)
y_predicted_probs = np.round(y_predicted_probs, 3)
# Note: Version 0.2.2 changed the following line to ensure probability labels are correct regardless of the order of target classes
columns = [f'{label_names[c]}_prob' for c in pipeline1.named_steps['classifier'].classes_ if c in target_classes]
predictions_df['predicted'] = predictions_df['predicted'].apply(lambda x: label_names[x])
predictions_df['true'] = predictions_df['true'].apply(lambda x: label_names[x])
predictions_df['correct'] = predictions_df['true'] == predictions_df['predicted']
predictions_df['text'] = X_test
predictions_df = pd.concat([predictions_df, pd.DataFrame(y_predicted_probs, columns=columns)], axis=1)

# output a preview of docs for each cell of confusion matrix ...
for true_target, target_name in enumerate(target_names):
    for predicted_target, target_name in enumerate(target_names):
        if true_target == predicted_target:
            print(f'\nCORRECTLY CLASSIFIED: {target_names[true_target]}')
        else:
            print(f'\n{target_names[true_target]} INCORRECTLY CLASSIFIED as: {target_names[predicted_target]}')
        print('=================================================================')

        display(predictions_df[(predictions_df['true'] == target_names[true_target]) & (predictions_df['predicted'] == target_names[predicted_target])])

pd.set_option('display.max_rows', 60) # setting back to the default


CORRECTLY CLASSIFIED: negative


,true,predicted,correct,text,negative_prob,neutral_prob,positive_prob
7,negative,negative,True,Omg this show is so predictable even for the 3rd ep. Rui En\u2019s ex boyfriend was framed for murder probably\u002c by the rich guy.,0.790,0.083,0.126
10,negative,negative,True,"@user so the thing next Thursday isn't free, you'd have to pay $15 to get in since you don't go to UMBC :/ and it ends at 11:30""",0.775,0.199,0.026
19,negative,negative,True,The sad part about this is tomorrow Nicki will be the angry black woman who went after poor white girl Miley,0.949,0.049,0.001
24,negative,negative,True,"""\""""@Bobbieliciouss: Don\u2019t think i\u2019m going to school tomorrow.\"""" School\u2019s gonna crack tomorrow""",0.413,0.377,0.209
45,negative,negative,True,@user Please refrain from interupting Dan's rants on Dana White with a stupid ass Sunday Countdown commercial... k? thanks!,0.956,0.030,0.014
...,...,...,...,...,...,...,...
1974,negative,negative,True,"""More like boring eagles""""""""""""""""@Tunnyking: C'mon bro, Go out and support the Super Eagles #RT @user I hate international breaks""",0.797,0.166,0.037
1985,negative,negative,True,Disappointed the Knicks vs Nets game got canceled tonight\u002c but I\u2019m even more hyped for Knicks vs Heat on Friday!,0.511,0.316,0.172
1989,negative,negative,True,"@user @user Islam is an Abrahamic faith, Andrew. It may make you feel a little uneasy but it's the same God you worship. Sorry.""",0.456,0.239,0.305
1992,negative,negative,True,kingpin Saudi Arabia posted a record $98 billion budget deficit in 2015 due to the sharp fall in oil prices finance ministry said on Monday,0.520,0.437,0.043



negative INCORRECTLY CLASSIFIED as: neutral


,true,predicted,correct,text,negative_prob,neutral_prob,positive_prob
2,negative,neutral,False,When girls become bandwagon fans of the Packers because of Harry. Do y'all even know who Aaron Rodgers is? Or what a 1st down is?,0.342,0.424,0.234
14,negative,neutral,False,F*** the hurricane party this Tues santospartyhaus w/ @user @user @user @ Santos Party House,0.040,0.614,0.346
21,negative,neutral,False,@user I haven't been able to watch TVD live these days due to Football. Every Thurs there is high school FB going on at 8pm. Like WTF!,0.191,0.597,0.212
60,negative,neutral,False,People are researching the Wolf crash to try and make SC racing safer. If anyone found his GoPro contact Me or Brian Sperry. It may help.,0.402,0.490,0.108
116,negative,neutral,False,"""Galaxy S II can be wiped by just clicking a link, other devices may be vulnerable #Galaxy #Samsung #HTML #Vulnerability""",0.319,0.527,0.154
...,...,...,...,...,...,...,...
1859,negative,neutral,False,"""George Lincoln Rockwell was one of the 1st to recognize that Conservatives like @user Buckley, Goldwater &amp; Reagan were #Cucks for Israel.""",0.373,0.520,0.108
1884,negative,neutral,False,Fox Chicago\u002c you make me angry. Playing the Vikings vs Redskins over the NFC Championship rematch 49ers vs Giants tomorrow????,0.254,0.451,0.295
1893,negative,neutral,False,"@user tom Brady did not deflate balls, but was suspended for 4 games bc he may or may not have known it was being done""",0.246,0.697,0.057
1910,negative,neutral,False,David Cameron's statement on camera on Thursday 03 September 2015: he will take in 'more' of the refugees: was he speaking TO TV Cameras?,0.223,0.629,0.149



negative INCORRECTLY CLASSIFIED as: positive


...

,true,predicted,correct,text,negative_prob,neutral_prob,positive_prob
6,positive,negative,False,#US 1st Lady Michelle Obama speaking at the 2015 Beating the Odds Summit to over 130 college-bound students at the pentagon office. &gt;&gt;,0.688,0.232,0.081
20,positive,negative,False,@user @user @user -YOU R THE BEST NEGOTIATOR.Wish it was U instead of Kerry that went.,0.367,0.298,0.335
43,positive,negative,False,I love how Thanksgiving break had hella bangers and we're in the 2nd week of winter break and have failed to have one yet,0.779,0.095,0.126
59,positive,negative,False,I need to see the 2nd episode of AHS now,0.468,0.210,0.322
68,positive,negative,False,i've made up mind. i'm gonna support both ot8 and Jessica equally. maybe the split is required to save their friendship. you may never know.,0.563,0.086,0.351
...,...,...,...,...,...,...,...
1914,positive,negative,False,"@user @user Funny how you call it """"Thursday"""" and don't believe Thor was real.""",0.443,0.354,0.203
1934,positive,negative,False,One of many things that Islam has taught me is to look upon charity as something that we all must do every day that the sun rises. Powerful.,0.422,0.352,0.226
1935,positive,negative,False,YOU MUST READ! Karina Smirnoff Wows in a Low-Cut Yellow Dress at the ESPY Awards (PHOTOS),0.504,0.335,0.161
1948,positive,negative,False,When I wake up tomorrow I'll be in a different country. Whoa! I didn't run into a David Beckham at the airport. That's a bummer.,0.540,0.102,0.358



positive INCORRECTLY CLASSIFIED as: neutral


,true,predicted,correct,text,negative_prob,neutral_prob,positive_prob
16,positive,neutral,False,Tom Brady is locked for Thursday. Let the season begin! #RepeatSeason,0.221,0.476,0.303
29,positive,neutral,False,"@user told you because you said """"Generally 15 August comes near Eid week"""". It happens for only 2 years after every 30 years. :)""",0.110,0.509,0.382
30,positive,neutral,False,Nicki did that for white media Idgaf . Nicki may act like she don't give af but she cares what the media thinks,0.293,0.689,0.018
33,positive,neutral,False,"""In the history of Tae Kwon Do, Chuck Norris was the first person from the West to be ranked a 8th Degree Black Belt Grand Master.""",0.075,0.760,0.165
36,positive,neutral,False,@user You may check with Amazon for Kindle as it's screen is glare free and can be read in Sunlight too.,0.271,0.457,0.272
...,...,...,...,...,...,...,...
1937,positive,neutral,False,"@user wow,ipad got just today free lol www.burna.in/fgye""",0.116,0.503,0.381
1946,positive,neutral,False,"""The sun shall not smite I by day, nor the moon by night"" Bob Marley &amp; the Wailers NIGHT SHIFT *Live in NYC 4/30/76*",0.214,0.520,0.266
1977,positive,neutral,False,Monday at Town Ballroom: RICHIE HAWTIN with LOCO DICE. Dude is so awesome. Tix still avail at,0.288,0.373,0.339
1986,positive,neutral,False,PM ready for reply on coal blocks: Congress: New Delhi\u002c Aug 22 (IANS) With the Bharatiya Janata Party (BJP)...,0.030,0.492,0.478



CORRECTLY CLASSIFIED: positive


,true,predicted,correct,text,negative_prob,neutral_prob,positive_prob
1,positive,positive,True,"""National hot dog day, national tequila day, then national dance day... Sounds like a Friday night.""",0.087,0.289,0.624
8,positive,positive,True,"""What a round by Paul Dunne, good luck tomorrow and I hope you win the Open.""",0.007,0.027,0.966
11,positive,positive,True,Can't wait for tomorrow at 9 pm! Chelsea v crystal palace! Hope the blues will win!,0.001,0.006,0.993
15,positive,positive,True,"""Thank you @user for the message. I'm very proud to be a Liverpudlian, may i get your follback? #LiverpudlianLoyalitasTanpaBatas #YNWA""",0.294,0.218,0.488
18,positive,positive,True,Eric Church is headlining a festival in Raleigh on October 17 &amp; 18 and I may or may not be the happiest person ever right now,0.109,0.208,0.684
...,...,...,...,...,...,...,...
1980,positive,positive,True,@user @user I think that was faster win then the rousey fight on Saturday Night!,0.072,0.423,0.505
1981,positive,positive,True,Sharknado 3 may be the best film I've seen yet. #Sharknado3 #America,0.118,0.236,0.645
1982,positive,positive,True,This Saturday &amp; Sunday come join us the @user at the Pomona Fairplex! Your ticket can WIN you a Brand New Car!,0.007,0.039,0.954
1994,positive,positive,True,Beautiful Bouquet with our Beautiful Bentley #bride #groom #wedding #wednesday #weddingcars #love #Repost...,0.015,0.067,0.918


In [231]:
selected_index = 1983

preview_row_text(predictions_df, selected_index, text_column = text_column, limit=400) # change limit to see more of the text if needed

,Value
Attribute,
true,neutral
predicted,positive
correct,False
negative_prob,0.115
neutral_prob,0.361
positive_prob,0.524


text:
Hey David Bowie Do u want to get iPh0ne 6 for FREE? U better check my bi0. Thx


### 4.3 Run inference on new (or old) data

You can also run inference on new data (or any of the texts from training/validation) by changing the contents of the `texts` list below. This outputs a prediction, the probabilities of each class and the features present within the text that are used by the model to make its predictions. The numbers for each feature are the input to the final step of the pipeline. They may be scaled or transformed depending on the pipeline components you've chosen.

## 4.3.1 Explore neutral INCORRECTLY CLASSIFIED as: negative

4 tweet examples were selected to present in the report. 

In [229]:
texts = ['''
Irving Plaza NYC Blackout Saturday night. Got limited spots left on the guest
list. Tweet me why you think you deserve them
''',
		'''
Chief Minister follows up on restoration work: Restoration of the damaged side
of the Anna arch began on Wednesday,...
''',
	'''
Digne and Falque caused Juventus real problems down their left in the 1st half.
#ASRoma #Juventus
''',
'''
Harper's Worst Offense against Refugees may be Climate Record as rising
temperatures add to chaos in the Middle East
''',
]

y_inference = pipeline1.predict(texts)

preprocessor = Pipeline(pipeline1.steps[:-1])
feature_names = preprocessor.named_steps['features'].get_feature_names_out()

for i, text in enumerate(texts):
	print(f"Text {i}: {text}")
	
	print(f"\tPredicted class: {label_names[y_inference[i]]}")
	print()

	y_inference_proba = pipeline1.predict_proba([text])
	
	# Note: Version 0.2.2 changed the following lines to ensure probability labels are correct regardless of the order of target classes
	for idx, prob in enumerate(y_inference_proba[0]):
		c = pipeline1.named_steps['classifier'].classes_[idx]
		if c in target_classes:
			print(f"\tProbability of class {label_names[c]}: {prob:.2f}")
	# End change for 0.2.2

	print()
	print("\tFeatures:")

	embeddings = 0
    
	frequencies = preprocessor.transform([text])
	if not isinstance(frequencies, np.ndarray):
		frequencies = frequencies.toarray()
	frequencies = frequencies[0].T
    
	for j, freq in enumerate(frequencies):
		if feature_names[j].startswith('embeddings_'):
			embeddings += 1
		elif freq > 0:
			print(f"\t{feature_names[j]}: {freq:.2f}")
	if embeddings > 0:
		print(f"\tFeatures also include {embeddings} embedding dimensions")

	print()


Text 0: 
Irving Plaza NYC Blackout Saturday night. Got limited spots left on the guest
list. Tweet me why you think you deserve them

	Predicted class: negative

	Probability of class negative: 0.83
	Probability of class neutral: 0.14
	Probability of class positive: 0.03

	Features:
	tokens__.: 2.47
	tokens__Saturday: 5.14
	tokens__Saturday night: 17.54
	tokens__left: 13.05
	tokens__night: 4.76
	tokens__night .: 14.56
	tokens__think: 6.71
	pos__PRON: 2.82
	pos__PROPN: 2.07
	pos__SCONJ: 1.90
	pos__VERB: 3.64
	lexicon__negative: 1.78

Text 1: 
Chief Minister follows up on restoration work: Restoration of the damaged side
of the Anna arch began on Wednesday,...

	Predicted class: neutral

	Probability of class negative: 0.44
	Probability of class neutral: 0.46
	Probability of class positive: 0.10

	Features:
	tokens__,: 2.40
	tokens__...: 3.87
	tokens__:: 3.92
	tokens__Wednesday: 12.95
	tokens__work: 14.86
	pos__PROPN: 1.66
	pos__VERB: 2.19
	lexicon__negative: 1.78

Text 2: 
Digne and Fal

## 4.3.2 Explore neutral INCORRECTLY CLASSIFIED as: positive

2 tweet examples were selected to present in the report. 

In [232]:
texts = ['''
My dads selling 4 tickets for the Jason Aldean, Tyler Farr and Cole Swidell
concert September 19th. $100 a piece. Dm me if interested!
''',
		'''
Hey David Bowie Do u want to get iPh0ne 6 for FREE? U better check my bi0. Thx
''',

]

y_inference = pipeline1.predict(texts)

preprocessor = Pipeline(pipeline1.steps[:-1])
feature_names = preprocessor.named_steps['features'].get_feature_names_out()

for i, text in enumerate(texts):
	print(f"Text {i}: {text}")
	
	print(f"\tPredicted class: {label_names[y_inference[i]]}")
	print()

	y_inference_proba = pipeline1.predict_proba([text])
	
	# Note: Version 0.2.2 changed the following lines to ensure probability labels are correct regardless of the order of target classes
	for idx, prob in enumerate(y_inference_proba[0]):
		c = pipeline1.named_steps['classifier'].classes_[idx]
		if c in target_classes:
			print(f"\tProbability of class {label_names[c]}: {prob:.2f}")
	# End change for 0.2.2

	print()
	print("\tFeatures:")

	embeddings = 0
    
	frequencies = preprocessor.transform([text])
	if not isinstance(frequencies, np.ndarray):
		frequencies = frequencies.toarray()
	frequencies = frequencies[0].T
    
	for j, freq in enumerate(frequencies):
		if feature_names[j].startswith('embeddings_'):
			embeddings += 1
		elif freq > 0:
			print(f"\t{feature_names[j]}: {freq:.2f}")
	if embeddings > 0:
		print(f"\tFeatures also include {embeddings} embedding dimensions")

	print()


Text 0: 
My dads selling 4 tickets for the Jason Aldean, Tyler Farr and Cole Swidell
concert September 19th. $100 a piece. Dm me if interested!

	Predicted class: positive

	Probability of class negative: 0.03
	Probability of class neutral: 0.44
	Probability of class positive: 0.54

	Features:
	tokens__!: 1.18
	tokens__$: 7.64
	tokens__,: 1.19
	tokens__.: 1.85
	tokens__4: 6.52
	tokens__Aldean: 15.82
	tokens__Jason: 11.99
	tokens__Jason Aldean: 15.82
	tokens__My: 7.21
	tokens__September: 8.14
	tokens__concert: 8.16
	tokens__tickets: 8.34
	pos__PRON: 1.41
	pos__PROPN: 2.90
	pos__SCONJ: 1.90
	pos__VERB: 1.46
	lexicon__positive: 1.18

Text 1: 
Hey David Bowie Do u want to get iPh0ne 6 for FREE? U better check my bi0. Thx

	Predicted class: positive

	Probability of class negative: 0.11
	Probability of class neutral: 0.36
	Probability of class positive: 0.52

	Features:
	tokens__.: 1.08
	tokens__6: 9.76
	tokens__?: 2.25
	tokens__David: 6.62
	tokens__Do: 10.95
	tokens__better: 8.71
	tokens__